In [2]:
!pwd

/Users/jahandadirfan/Documents/GitHub/BOL-LPP-Reproduction/notebooks


In [4]:
import gridstatus
import pandas as pd
from pathlib import Path
import requests

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

ercot = gridstatus.Ercot()

# --- Price data ---
years = range(2013, 2019)
price_frames = []
for year in years:
    print(f"Downloading DAM SPP for {year}...")
    price_frames.append(ercot.get_dam_spp(year, verbose=True))

dam_spp_all = pd.concat(price_frames, ignore_index=True)

zone_mapping = {"LZ_NORTH": "North", "LZ_SOUTH": "South", "LZ_HOUSTON": "Coast"}
dam_spp_zones = dam_spp_all[dam_spp_all["Location"].isin(zone_mapping.keys())].copy()
dam_spp_zones["Zone"] = dam_spp_zones["Location"].map(zone_mapping)
dam_spp_zones.to_csv(RAW_DIR / "ercot_dam_spp_zones_2013_2018.csv", index=False)
print("Price done:", dam_spp_zones.shape)

# --- Load data ---
load_frames = []
for year in years:
    print(f"Downloading hourly load for {year}...")
    load_frames.append(ercot.get_hourly_load_post_settlements(
        date=f"{year}-01-01", end=f"{year}-12-31", verbose=True))

load_all = pd.concat(load_frames, ignore_index=True)

# Map ERCOT weather zones to the paper's load-zone names (Option A: single best match)
weather_to_load_zone = {
    "North Central": "North",    # DFW metroplex -> LZ_NORTH
    "South Central": "South",    # San Antonio/Austin -> LZ_SOUTH
    "Coast": "Coast",            # Houston -> LZ_HOUSTON
}

load_long = load_all.melt(
    id_vars=["Interval Start", "Interval End"],
    value_vars=list(weather_to_load_zone.keys()),
    var_name="WeatherZone", value_name="Load_MW")

load_long["Zone"] = load_long["WeatherZone"].map(weather_to_load_zone)
load_long = load_long.drop(columns=["WeatherZone"])

load_long.to_csv(RAW_DIR / "ercot_hourly_load_zones_2013_2018.csv", index=False)
print("Load done:", load_long.shape)
print(load_long.groupby("Zone")["Load_MW"].mean().round(0))

# --- Weather data ---
LAT, LON = 32.90, -97.04
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LAT, "longitude": LON,
    "start_date": "2013-01-01", "end_date": "2018-12-31",
    "hourly": "temperature_2m,dewpoint_2m,windspeed_10m,winddirection_10m",
    "timezone": "America/Chicago",
}
response = requests.get(url, params=params)
response.raise_for_status()
weather_df = pd.DataFrame(response.json()["hourly"])
weather_df.to_csv(RAW_DIR / "weather_north_2013_2018.csv", index=False)
print("Weather done:", weather_df.shape)

print("\nALL FILES:", [f.name for f in RAW_DIR.glob("*.csv")])

2026-09-04 21:32:56 - INFO - Fetching document https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=13060&_1788539576
2026-09-04 21:32:56 - INFO - Requesting https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=13060&_1788539576 with {}


Requesting https://www.ercot.com/misdownload/servlets/mirDownload?doclookupId=387408239


2026-09-04 21:33:00 - INFO - Fetching document https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=10008&_1788539580
2026-09-04 21:33:00 - INFO - Requesting https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=10008&_1788539580 with {}
2026-09-04 21:33:01 - INFO - Fetching https://www.ercot.com/misdownload/servlets/mirDownload?doclookupId=1269508726
2026-09-04 21:33:01 - INFO - Fetching document https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=13060&_1788539581
2026-09-04 21:33:01 - INFO - Requesting https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=13060&_1788539581 with {}


Requesting https://www.ercot.com/misdownload/servlets/mirDownload?doclookupId=468450666


2026-09-04 21:33:05 - INFO - Fetching document https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=10008&_1788539585
2026-09-04 21:33:05 - INFO - Requesting https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=10008&_1788539585 with {}
2026-09-04 21:33:05 - INFO - Fetching https://www.ercot.com/misdownload/servlets/mirDownload?doclookupId=1269508726
2026-09-04 21:33:06 - INFO - Fetching document https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=13060&_1788539586
2026-09-04 21:33:06 - INFO - Requesting https://www.ercot.com/misapp/servlets/IceDocListJsonWS?reportTypeId=13060&_1788539586 with {}


...

2026-09-04 21:33:32 - DEBUG - Changing timezone for DST duplicate at 2015-11-01 01:00:00-06:00
2026-09-04 21:33:32 - INFO - Fetching historical load data for year 2016


2026-09-04 21:33:33 - DEBUG - Changing timezone for DST duplicate at 2016-11-06 01:00:00-06:00
2026-09-04 21:33:33 - INFO - Fetching historical load data for year 2017


2026-09-04 21:33:34 - DEBUG - Changing timezone for DST duplicate at 2017-11-05 01:00:00-06:00
2026-09-04 21:33:34 - INFO - Fetching historical load data for year 2018


2026-09-04 21:33:35 - DEBUG - Changing timezone for DST duplicate at 2018-11-04 01:00:00-06:00


Load done: (157752, 4)
Zone
Coast    11498.0
North    13116.0
South     6508.0
Name: Load_MW, dtype: float64
Weather done: (52584, 5)

ALL FILES: ['ercot_hourly_load_zones_2013_2018.csv', 'ercot_dam_spp_zones_2013_2018.csv', 'weather_north_2013_2018.csv']
